# DC-AE 重建评估

加载训练好的 DC-AE，在指定 split 上计算重建指标（MSE / MAE / RMSE / R² / PSNR）并可视化原场、重建场与误差场。

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from model.dcae import DCAE
from data.dataset import build_dataset, load_constants
from data.data_utils import normalize_fn
from config import get_dataset_config

DEVICE = torch.device('cpu')  # DCAE 参数量大，CPU 评估避免显存溢出

# ── 可修改配置 ──────────────────────────────────────────────────────────────
DATA_NAME   = 'glorys12_kuroshio_extension'
TAG         = 'dcae_bc64_cm1248_lc16_fft0.5'
SPLIT       = 'val'   # train / val / test
BATCH_SIZE  = 4
NUM_WORKERS = 4
MAX_BATCHES = 10      # None = 全量；与 eval_vqvae 保持一致

# DC-AE 结构（与训练保持一致）
BASE_CHANNELS         = 64
CHANNEL_MULTIPLIERS   = [1, 2, 4, 8]
LATENT_CHANNELS       = 16
NUM_RES_BLOCKS        = 2
ATTENTION_RESOLUTIONS = [1, 2]
NUM_HEADS             = 8
# ────────────────────────────────────────────────────────────────────────────

CKPT_PATH = os.path.join(PROJECT_ROOT, 'output', DATA_NAME, TAG, 'best_model.pth')
assert os.path.exists(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DEVICE:       {DEVICE}')
print(f'CKPT_PATH:    {CKPT_PATH}')

In [ ]:
# ── 构建数据集与常量 ─────────────────────────────────────────────────────────
dataset_config = get_dataset_config(DATA_NAME)

_date_ranges = {
    'train': dataset_config.train_date_range,
    'val':   dataset_config.val_date_range,
    'test':  dataset_config.test_date_range,
}
dataset = build_dataset(dataset_config.raw_data_dir, _date_ranges[SPLIT])
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,   # shuffle=False，与 eval_vqvae 一致
                     num_workers=NUM_WORKERS, pin_memory=False)

# constants = (normed_ocean_mean, normed_ocean_std, raw_ocean_min, raw_ocean_max, depths, mask)
constants = load_constants(dataset_config.constant_dir)
mu    = constants[0][..., None, None].to(DEVICE)  # [C, 1, 1]
sigma = constants[1][..., None, None].to(DEVICE)  # [C, 1, 1]
mask  = constants[-1].float()                      # [C, H, W]  (1=ocean, 0=land)

print(f'Dataset [{SPLIT}]: {len(dataset)} samples')
print(f'Channels: {dataset_config.num_channels},  mask shape: {tuple(mask.shape)}')
print(f'mu shape: {tuple(mu.shape)},  sigma shape: {tuple(sigma.shape)}')

In [ ]:
# ── 构建并加载模型 ───────────────────────────────────────────────────────────
def build_and_load_model(ckpt_path, in_channels, device):
    ckpt  = torch.load(ckpt_path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt.get('model', ckpt))
    # 去除 DDP 的 'module.' 前缀
    state = {(k[7:] if k.startswith('module.') else k): v for k, v in state.items()}

    model = DCAE(
        in_channels=in_channels,
        base_channels=BASE_CHANNELS,
        channel_multipliers=CHANNEL_MULTIPLIERS,
        latent_channels=LATENT_CHANNELS,
        num_res_blocks=NUM_RES_BLOCKS,
        attention_resolutions=ATTENTION_RESOLUTIONS,
        num_heads=NUM_HEADS,
    ).to(device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

model = build_and_load_model(CKPT_PATH, dataset_config.num_channels, DEVICE)
print(f'Model loaded.  Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 评估工具函数 ─────────────────────────────────────────────────────────────
def masked_metrics(pred, target, mask):
    """MSE / MAE / RMSE / R² / PSNR，只在掩膜有效（ocean）像素上计算。

    pred, target : [B, C, H, W]  float32，无 NaN
    mask         : 可广播到 [B, C, H, W]，0/1
    返回长度为 B 的 list of dict。
    """
    eps = 1e-8
    m = mask.to(pred.device)
    while m.ndim < pred.ndim:
        m = m.unsqueeze(0)
    m = m.expand_as(pred)

    B     = pred.shape[0]
    flat  = lambda t: t.reshape(B, -1)

    n_valid  = flat(m).sum(1).clamp_min(eps)                # [B]
    diff     = (pred - target) * m
    sq_err   = flat(diff.pow(2)).sum(1)                     # [B]
    abs_err  = flat(diff.abs()).sum(1)                      # [B]
    mse      = sq_err / n_valid
    mae      = abs_err / n_valid
    rmse     = mse.sqrt()

    # R²：ss_tot 仅统计 ocean 像素的方差
    mean_t  = flat(target * m).sum(1) / n_valid             # [B]
    ss_tot  = flat((target - mean_t.view(B, 1, 1, 1)).pow(2) * m).sum(1).clamp_min(eps)
    r2      = 1.0 - sq_err / ss_tot

    # PSNR：数据范围取 ocean 像素的 max-min
    INF     = 1e10
    oc_max  = (target * m + (1 - m) * (-INF)).reshape(B, -1).max(1).values
    oc_min  = (target * m + (1 - m) *   INF ).reshape(B, -1).min(1).values
    d_range = (oc_max - oc_min).clamp_min(eps)
    psnr    = 20.0 * torch.log10(d_range / rmse.clamp_min(eps))

    return [
        {'mse':  mse[i].item(),  'mae':  mae[i].item(),
         'rmse': rmse[i].item(), 'r2':   r2[i].item(),
         'psnr': psnr[i].item()}
        for i in range(B)
    ]


@torch.inference_mode()
def evaluate(model, loader, mu, sigma, mask, device, max_batches=None):
    """遍历数据集，返回 (per-sample metrics DataFrame, vis_cache)。"""
    records   = []
    vis_cache = None

    for step, batch in enumerate(loader):
        if max_batches is not None and step >= max_batches:
            break

        raw = batch.to(device=device, dtype=torch.float32)

        # 归一化（land NaN → 0，再做 z-score）
        x_norm = normalize_fn(raw, mu=mu, sigma=sigma)

        # 前向推理
        recon_norm = model(x_norm)

        # 反归一化到物理空间（land 像素 ≈ per-channel mean，被 mask 屏蔽）
        x_phys     = (x_norm     * sigma + mu).cpu()
        recon_phys = (recon_norm * sigma + mu).cpu()

        for rec in masked_metrics(recon_phys, x_phys, mask):
            rec['step'] = step
            records.append(rec)

        if vis_cache is None:
            vis_cache = {'x_phys': x_phys.clone(), 'recon_phys': recon_phys.clone()}

    return pd.DataFrame(records), vis_cache

In [ ]:
# ── 运行评估 ─────────────────────────────────────────────────────────────────
df, vis_cache = evaluate(model, loader, mu, sigma, mask, DEVICE, max_batches=MAX_BATCHES)

metric_cols = ['mse', 'mae', 'rmse', 'r2', 'psnr']
print(f'Evaluated {len(df)} samples\n')
print('Mean metrics:')
print(df[metric_cols].mean().to_frame('mean').T.round(4).to_string())
df.head(10)

In [ ]:
# ── 单样本数值范围检查 ────────────────────────────────────────────────────────
x_phys     = vis_cache['x_phys']      # [B, C, H, W]
recon_phys = vis_cache['recon_phys']
mask_bool  = mask.bool()              # [C, H, W]
SAMPLE_IDX = 0

for ch in [0, 25, 50, 75, 100]:
    if ch >= x_phys.shape[1]:
        break
    mk     = mask_bool[ch] if mask_bool.ndim == 3 else mask_bool
    gt_oc  = x_phys[SAMPLE_IDX, ch][mk]
    rc_oc  = recon_phys[SAMPLE_IDX, ch][mk]
    print(
        f'CH{ch:3d}  input [{gt_oc.min():.3f}, {gt_oc.max():.3f}]  '
        f'recon [{rc_oc.min():.3f}, {rc_oc.max():.3f}]  '
        f'MAE={torch.mean(torch.abs(rc_oc - gt_oc)):.4f}'
    )

In [ ]:
# ── 指标分布直方图 ───────────────────────────────────────────────────────────
from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 5, figsize=(20, 3))
for ax, col in zip(axes, ['mse', 'mae', 'rmse', 'r2', 'psnr']):
    ax.hist(df[col].dropna().values, bins=20)
    ax.set_title(col)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# ── 重建可视化：原场 / 重建场 / 绝对误差 ─────────────────────────────────────
x_np     = vis_cache['x_phys'].numpy()      # [B, C, H, W]
recon_np = vis_cache['recon_phys'].numpy()
mask_np  = mask.numpy().astype(bool)        # [C, H, W]
SAMPLE_IDX = 0

max_ch   = x_np.shape[1] - 1
CHANNELS = sorted(set([0, max_ch // 4, max_ch // 2, 3 * max_ch // 4, max_ch]))

def get_mask2d(ch):
    return mask_np[ch] if mask_np.ndim == 3 else mask_np

def masked_field(arr, ch):
    """NaN at land pixels for display."""
    out = arr[SAMPLE_IDX, ch].copy()
    out[~get_mask2d(ch)] = np.nan
    return out

fig, axes = plt.subplots(len(CHANNELS), 3, figsize=(12, 3 * len(CHANNELS)))
if len(CHANNELS) == 1:
    axes = axes[np.newaxis]

for r, ch in enumerate(CHANNELS):
    gt_d  = masked_field(x_np, ch)
    rc_d  = masked_field(recon_np, ch)
    err_d = np.abs(rc_d - gt_d)

    shared_min = np.nanmin(np.stack([gt_d, rc_d]))
    shared_max = np.nanmax(np.stack([gt_d, rc_d]))
    err_max = np.nanpercentile(err_d, 99) if np.isfinite(err_d).any() else 1.0
    err_max = max(float(err_max), 1e-8)

    for ax, data, title, cmap, vmi, vma in [
        (axes[r, 0], gt_d,  f'CH{ch} Input(denorm)',  'viridis', shared_min, shared_max),
        (axes[r, 1], rc_d,  f'CH{ch} Recon(denorm)',  'viridis', shared_min, shared_max),
        (axes[r, 2], err_d, f'CH{ch} Abs Error(denorm)', 'viridis', 0,          err_max),
    ]:
        im = ax.imshow(data, vmin=vmi, vmax=vma, cmap=cmap)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()